# Phase 2.4 — Analyse ANCOVA à trois bras

**Projet** : RCT Foyers Ameliores — Senegal rural

Specification ANCOVA par outcome :

    Y_E = a + b1*T1 + b2*T2 + g*Y_B + d*strate + eps

- **b1** = impact T1 vs T3 (foyer + formation vs controle)
- **b2** = impact T2 vs T3 (formation seule vs controle)
- **b1-b2** = effet marginal du foyer (T1 vs T2)

SE heteroskedastiques robustes (HC3). Stratification incluant les zones
agro-ecologiques (CENTRE NORD + CENTRE OUEST fusionnes en CENTRE pour
corriger un desequilibre d'assignation).

## 1. Setup et preparation des donnees

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import statsmodels.formula.api as smf
from statsmodels.stats.contrast import ContrastResults
from scipy import stats

pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.dpi'] = 100

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'python' else Path.cwd()
PROC = ROOT / 'data' / 'processed'
FIG  = ROOT / 'docs' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(PROC / 'panel_ancova.parquet')
print(f"Panel : {df.shape[0]} menages x {df.shape[1]} variables")
print(f"Traitement : {df['treatment'].value_counts().to_dict()}")

In [ ]:
# Fusion des strates CENTRE NORD + CENTRE OUEST
# Diagnostic : CENTRE OUEST contient uniquement T3 (75 menages),
# ce qui creerait une colinearite parfaite avec T3 dans le modele a effets fixes.
# CENTRE NORD a T1=124, T2=124, T3=50. Fusionne : T1=124, T2=124, T3=125 (equilibre).
df['strate'] = df['zone'].replace({
    'CENTRE NORD' : 'CENTRE',
    'CENTRE OUEST': 'CENTRE',
})
print("Strates apres fusion :")
print(df.groupby('strate')['treatment'].value_counts().unstack(fill_value=0).to_string())

# Dummies T1, T2 (T3 = reference)
assert (df['T1'] + df['T2']).max() <= 1, "Un menage ne peut pas etre T1 ET T2"

# Reference de strate = CENTRE SUD (la plus grande et la plus equilibree)
STRATE_REF = 'CENTRE SUD'
df['strate_cat'] = pd.Categorical(df['strate'],
                                   categories=['CENTRE SUD','CENTRE','EST','NORD','SUD'])
print(f"\nCategorie de reference pour les effets fixes de strate : {STRATE_REF}")

## 2. Fonction ANCOVA

In [ ]:
def ancova(Y_endline, Y_baseline, data=df,
           cluster_col='village', verbose=False):
    """
    Estime :  Y_E = a + b1*T1 + b2*T2 + g*Y_B + d*strate + eps
    Retourne un dict avec les trois estimands (b1, b2, b1-b2) + leurs SE et IC95.
    """
    cols_needed = ['T1','T2','strate_cat', Y_endline]
    if Y_baseline and Y_baseline in data.columns:
        cols_needed.append(Y_baseline)

    sub = data[cols_needed + [cluster_col]].dropna()
    n   = len(sub)
    n_T1, n_T2, n_T3 = (sub['T1']==1).sum(), (sub['T2']==1).sum(),                         ((sub['T1']==0)&(sub['T2']==0)).sum()

    if n < 100:
        return None

    # Formule ANCOVA
    if Y_baseline and Y_baseline in sub.columns and sub[Y_baseline].notna().all():
        formula = f'{Y_endline} ~ T1 + T2 + {Y_baseline} + C(strate_cat)'
    else:
        formula = f'{Y_endline} ~ T1 + T2 + C(strate_cat)'

    m = smf.ols(formula, data=sub).fit(
        cov_type='cluster',
        cov_kwds={'groups': sub[cluster_col]}
    )

    # --- Extraction des trois estimands ---
    results = {}

    for arm, name in [('T1','T1_vs_T3'), ('T2','T2_vs_T3')]:
        coef = m.params.get(arm, np.nan)
        se   = m.bse.get(arm,   np.nan)
        pval = m.pvalues.get(arm, np.nan)
        results[name] = {
            'coef': coef, 'se': se, 'pval': pval,
            'lo': coef - 1.96*se, 'hi': coef + 1.96*se
        }

    # b1 - b2 via test de Wald manuel (plus robuste que t_test avec cluster SE)
    try:
        b1_est = m.params.get('T1', np.nan)
        b2_est = m.params.get('T2', np.nan)
        diff   = b1_est - b2_est
        cov    = m.cov_params()
        var_d  = (cov.loc['T1','T1'] + cov.loc['T2','T2']
                  - 2*cov.loc['T1','T2'])
        se_d   = float(np.sqrt(max(var_d, 0)))
        t_stat = diff / se_d if se_d > 0 else np.nan
        pv_d   = float(2 * (1 - stats.t.cdf(abs(t_stat), df=m.df_resid)))
        results['T1_vs_T2'] = {
            'coef': diff, 'se': se_d, 'pval': pv_d,
            'lo': diff - 1.96*se_d, 'hi': diff + 1.96*se_d
        }
    except Exception as ex:
        results['T1_vs_T2'] = {'coef': np.nan, 'se': np.nan,
                                'pval': np.nan, 'lo': np.nan, 'hi': np.nan}

    # Statistiques descriptives
    g = sub.groupby('T1')  # rough: T1 vs rest
    mean_T3 = sub.loc[(sub['T1']==0)&(sub['T2']==0), Y_endline].mean()
    mean_T1 = sub.loc[sub['T1']==1, Y_endline].mean()
    mean_T2 = sub.loc[sub['T2']==1, Y_endline].mean()
    sd_ctrl  = sub.loc[(sub['T1']==0)&(sub['T2']==0), Y_endline].std()

    results['meta'] = {
        'n': n, 'n_T1': n_T1, 'n_T2': n_T2, 'n_T3': n_T3,
        'mean_T3': mean_T3, 'mean_T1': mean_T1, 'mean_T2': mean_T2,
        'sd_ctrl': sd_ctrl,
        'formula': formula,
        'Y_E': Y_endline, 'Y_B': Y_baseline,
    }

    if verbose:
        print(m.summary2().tables[1][['Coef.','Std.Err.','P>|t|']])

    return results


def std_effect(coef, sd_ctrl):
    """Effet standardise (Cohen d-like) vs distribution controle endline."""
    if sd_ctrl and sd_ctrl > 0:
        return coef / sd_ctrl
    return np.nan

print("Fonction ancova() definie.")

## 3. Resultats par question analytique

### Q1 — Temps de travail domestique

In [ ]:
# Outcome principal : taches menageres par jour (h) de la femme
# Baseline control : taches_menage_h_mean_B
res_q1 = {}

outcomes_q1 = [
    ('taches_menage_h_mean_E',     'taches_menage_h_mean_B',     'Taches menageres / jour (h) — femme moy.'),
    ('travail_propre_compte_h_mean_E','travail_propre_compte_h_mean_B','Travail propre compte / jour (h)'),
]

for Y_E, Y_B, label in outcomes_q1:
    if Y_E not in df.columns: continue
    r = ancova(Y_E, Y_B)
    if r:
        res_q1[label] = r
        d = r['meta']
        b1 = r['T1_vs_T3']
        b2 = r['T2_vs_T3']
        bd = r['T1_vs_T2']
        print(f"\n=== {label} (n={d['n']}) ===")
        print(f"  Moy. T3 (ctrl) : {d['mean_T3']:.3f} h/j")
        print(f"  b1 (T1 vs T3)  : {b1['coef']:+.3f}  SE={b1['se']:.3f}  p={b1['pval']:.3f}  [{b1['lo']:.3f}; {b1['hi']:.3f}]")
        print(f"  b2 (T2 vs T3)  : {b2['coef']:+.3f}  SE={b2['se']:.3f}  p={b2['pval']:.3f}  [{b2['lo']:.3f}; {b2['hi']:.3f}]")
        print(f"  b1-b2 (foyer)  : {bd['coef']:+.3f}  SE={bd['se']:.3f}  p={bd['pval']:.3f}  [{bd['lo']:.3f}; {bd['hi']:.3f}]")
        ef1 = std_effect(b1['coef'], d['sd_ctrl'])
        print(f"  Effet std. b1  : {ef1:+.3f} SD")

### Q2 — Revenu

In [ ]:
outcomes_q2 = [
    ('revenu_menage_E',  'revenu_menage_B',  'Revenu total menage (FCFA/mois)'),
    ('revenu_femme_E',   'revenu_femme_B',   'Revenu femme ciblee (FCFA/mois)'),
]
res_q2 = {}
for Y_E, Y_B, label in outcomes_q2:
    if Y_E not in df.columns: continue
    r = ancova(Y_E, Y_B)
    if r:
        res_q2[label] = r
        d, b1, b2, bd = r['meta'], r['T1_vs_T3'], r['T2_vs_T3'], r['T1_vs_T2']
        print(f"\n=== {label} (n={d['n']}) ===")
        print(f"  Moy. T3 (ctrl) : {d['mean_T3']:,.0f}")
        print(f"  b1 (T1 vs T3)  : {b1['coef']:+,.0f}  SE={b1['se']:,.0f}  p={b1['pval']:.3f}")
        print(f"  b2 (T2 vs T3)  : {b2['coef']:+,.0f}  SE={b2['se']:,.0f}  p={b2['pval']:.3f}")
        print(f"  b1-b2 (foyer)  : {bd['coef']:+,.0f}  SE={bd['se']:,.0f}  p={bd['pval']:.3f}")
        print(f"  Effet std. b1  : {std_effect(b1['coef'], d['sd_ctrl']):+.3f} SD")

### Q3 — Attitude entrepreneuriale

In [ ]:
outcomes_q3 = [
    ('score_O_global_E',       'score_O_global_B',       'Score O global (O1-O30)'),
    ('score_entrepreneurial_E','score_entrepreneurial_B', 'Score motivation travail (O1-O9)'),
    ('score_genre_attitudes_E','score_genre_attitudes_B', 'Score attitudes genre (O19-O30)'),
]
res_q3 = {}
for Y_E, Y_B, label in outcomes_q3:
    if Y_E not in df.columns: continue
    r = ancova(Y_E, Y_B)
    if r:
        res_q3[label] = r
        d, b1, b2, bd = r['meta'], r['T1_vs_T3'], r['T2_vs_T3'], r['T1_vs_T2']
        print(f"\n=== {label} (n={d['n']}) ===")
        print(f"  Moy. T3  : {d['mean_T3']:.3f} | b1: {b1['coef']:+.4f} (p={b1['pval']:.3f}) | b2: {b2['coef']:+.4f} (p={b2['pval']:.3f}) | b1-b2: {bd['coef']:+.4f} (p={bd['pval']:.3f})")
        print(f"  Effet std. b1: {std_effect(b1['coef'], d['sd_ctrl']):+.3f} SD")

### Q4 — Combustibles et bien-etre

In [ ]:
outcomes_q4 = [
    ('D56_charbon_E', 'D56_charbon_B', 'Stock charbon (FCFA)'),
    ('D56_bois_E',    'D56_bois_B',    'Stock bois (FCFA)'),
    ('D56_gaz_E',     'D56_gaz_B',     'Stock gaz (FCFA)'),
    ('N17_E',         'N17_B',         'Echelle bien-etre (1-6)'),
]
res_q4 = {}
for Y_E, Y_B, label in outcomes_q4:
    if Y_E not in df.columns: continue
    r = ancova(Y_E, Y_B)
    if r:
        res_q4[label] = r
        d, b1, b2, bd = r['meta'], r['T1_vs_T3'], r['T2_vs_T3'], r['T1_vs_T2']
        print(f"\n=== {label} (n={d['n']}) ===")
        print(f"  Moy. T3  : {d['mean_T3']:.3f}")
        print(f"  b1: {b1['coef']:+.4f} (p={b1['pval']:.3f}) | b2: {b2['coef']:+.4f} (p={b2['pval']:.3f}) | b1-b2: {bd['coef']:+.4f} (p={bd['pval']:.3f})")

## 4. Tableau recapitulatif

In [ ]:
all_results = {}
all_results.update(res_q1)
all_results.update(res_q2)
all_results.update(res_q3)
all_results.update(res_q4)

rows = []
for label, r in all_results.items():
    if not r: continue
    d   = r['meta']
    b1  = r['T1_vs_T3']
    b2  = r['T2_vs_T3']
    bd  = r['T1_vs_T2']
    rows.append({
        'Outcome'        : label,
        'N'              : d['n'],
        'Moy. T3'        : round(d['mean_T3'],3),
        'b1 T1vsT3'      : round(b1['coef'],3),
        'SE b1'          : round(b1['se'],3),
        'p b1'           : round(b1['pval'],3),
        'Sig. b1'        : '*' if b1['pval']<0.05 else ('†' if b1['pval']<0.10 else ''),
        'b2 T2vsT3'      : round(b2['coef'],3),
        'SE b2'          : round(b2['se'],3),
        'p b2'           : round(b2['pval'],3),
        'Sig. b2'        : '*' if b2['pval']<0.05 else ('†' if b2['pval']<0.10 else ''),
        'b1-b2 (foyer)'  : round(bd['coef'],3),
        'p (foyer)'      : round(bd['pval'],3),
        'Sig. foyer'     : '*' if bd['pval']<0.05 else ('†' if bd['pval']<0.10 else ''),
        'Effet std. b1'  : round(std_effect(b1['coef'], d['sd_ctrl']),3),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
summary.to_csv(PROC / 'agg_ancova_results.csv', index=False, encoding='utf-8')
print("\nSauvegarde : data/processed/agg_ancova_results.csv")

## 5. Forest plot

In [ ]:
# Forest plot des effets standardises (Cohen d-like) — 3 estimands
# Une sous-figure par question analytique
questions = [
    ('Q1 — Temps de travail', res_q1),
    ('Q2 — Revenu',           res_q2),
    ('Q3 — Attitude entrepr.', res_q3),
    ('Q4 — Combustibles / BE', res_q4),
]
questions = [(q, r) for q, r in questions if r]

COLORS = {'T1 vs T3': '#d62728', 'T2 vs T3': '#1f77b4', 'T1 vs T2 (foyer)': '#2ca02c'}
OFFSETS = {'T1 vs T3': -0.22, 'T2 vs T3': 0, 'T1 vs T2 (foyer)': 0.22}

fig, axes = plt.subplots(1, len(questions), figsize=(5*len(questions), max(4, 0.55*max(len(r) for _, r in questions)+2)))

for ax, (q_label, q_res) in zip(axes if len(questions)>1 else [axes], questions):
    labels = list(q_res.keys())
    y = np.arange(len(labels))

    for estimand, key, color in [
        ('T1 vs T3', 'T1_vs_T3', COLORS['T1 vs T3']),
        ('T2 vs T3', 'T2_vs_T3', COLORS['T2 vs T3']),
        ('T1 vs T2 (foyer)', 'T1_vs_T2', COLORS['T1 vs T2 (foyer)']),
    ]:
        offset = OFFSETS[estimand]
        ys = y + offset
        coefs = []
        los   = []
        his   = []
        for lab in labels:
            r = q_res[lab]
            d = r['meta']
            coef = std_effect(r[key]['coef'], d['sd_ctrl'])
            lo   = std_effect(r[key]['lo'],   d['sd_ctrl'])
            hi   = std_effect(r[key]['hi'],   d['sd_ctrl'])
            coefs.append(coef if not np.isnan(coef) else 0)
            los.append(lo if not np.isnan(lo) else 0)
            his.append(hi if not np.isnan(hi) else 0)

        ax.errorbar(coefs, ys,
                    xerr=[np.array(coefs)-np.array(los),
                          np.array(his)-np.array(coefs)],
                    fmt='o', color=color, label=estimand,
                    capsize=3, ms=5, linewidth=1.2, zorder=3)

    ax.axvline(0, color='black', lw=0.8)
    ax.axvline( 0.2, color='grey', lw=0.5, ls='--', alpha=0.5)
    ax.axvline(-0.2, color='grey', lw=0.5, ls='--', alpha=0.5)
    ax.set_yticks(y)
    short_labels = [l[:35] for l in labels]
    ax.set_yticklabels(short_labels, fontsize=9)
    ax.set_title(q_label, fontweight='bold', fontsize=10)
    ax.set_xlabel('Effet standardise (b/SD controle)', fontsize=9)
    if ax == (axes[0] if len(questions)>1 else axes):
        ax.legend(fontsize=8, loc='lower right')

plt.suptitle("Effets ANCOVA — Impact projet foyers ameliores\n"
             "Rouge=T1vsT3 (total) · Bleu=T2vsT3 (formation) · Vert=T1vsT2 (foyer seul)",
             fontsize=10, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG / 'forest_plot_ancova.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Verification de robustesse — sans controle baseline

In [ ]:
# ANCOVA sans Y_B (equivalent a une simple comparaison endline par bras)
# Si les resultats sont similaires -> la randomisation a bien equilibre, le
# baseline control ajoute seulement de la precision, pas de correction de biais.
print("=== Robustesse : ANCOVA sans controle baseline ===")
for Y_E, Y_B, label in (list(outcomes_q1) + [outcomes_q2[0]] +
                         [outcomes_q3[0]] + [outcomes_q4[3]]):
    if Y_E not in df.columns: continue
    r_with = ancova(Y_E, Y_B)
    r_no   = ancova(Y_E, None)
    if r_with and r_no:
        diff_b1 = r_with['T1_vs_T3']['coef'] - r_no['T1_vs_T3']['coef']
        print(f"{Y_E[:30]:30s}  b1_avec={r_with['T1_vs_T3']['coef']:+.3f}  "
              f"b1_sans={r_no['T1_vs_T3']['coef']:+.3f}  "
              f"delta={diff_b1:+.3f}  "
              f"({'stable' if abs(diff_b1/max(abs(r_with['T1_vs_T3']['coef']),0.001))<0.15 else 'ATTENTION'})")

## 7. Export Power BI

In [ ]:
# Format long pour le dashboard Power BI
rows_pbi = []
QUESTION_MAP = {**{k: 'Q1' for k in res_q1},
                **{k: 'Q2' for k in res_q2},
                **{k: 'Q3' for k in res_q3},
                **{k: 'Q4' for k in res_q4}}

for label, r in all_results.items():
    if not r: continue
    d = r['meta']
    for estimand, key, comp in [
        ('T1 vs T3 (total)',    'T1_vs_T3', 'T1vsT3'),
        ('T2 vs T3 (formation)','T2_vs_T3', 'T2vsT3'),
        ('T1 vs T2 (foyer)',    'T1_vs_T2', 'T1vsT2'),
    ]:
        e = r[key]
        rows_pbi.append({
            'question'      : QUESTION_MAP.get(label, ''),
            'outcome'       : label,
            'comparaison'   : estimand,
            'coef'          : round(e['coef'], 4),
            'se'            : round(e['se'],   4),
            'pval'          : round(e['pval'], 4),
            'lo95'          : round(e['lo'],   4),
            'hi95'          : round(e['hi'],   4),
            'effet_std'     : round(std_effect(e['coef'], d['sd_ctrl']), 4),
            'mean_T3'       : round(d['mean_T3'], 4),
            'n'             : d['n'],
            'significatif'  : 1 if e['pval'] < 0.05 else 0,
        })

pbi_df = pd.DataFrame(rows_pbi)
pbi_df.to_parquet(PROC / 'agg_impact_estimates.parquet', index=False)
print(f"agg_impact_estimates.parquet : {len(pbi_df)} lignes")
print(pbi_df[['outcome','comparaison','coef','pval','significatif']].to_string(index=False))

## 8. Synthese

Documenter ici apres lecture des resultats :

**Interpretation des estimands** :
- b1 (T1 vs T3) = impact du PAQUET complet (foyer + formation)
- b2 (T2 vs T3) = impact de la formation seule
- b1-b2 = impact marginal du foyer amliore CONDITIONNEL a la formation

**Lecture attendue sous la theorie du changement** :
- Q1 (temps) : b1 < 0 si le foyer reduit les taches menageres
- Q1 (travail compte propre) : b1 > 0 si le temps libere est reinvesti economiquement  
- Q2 (revenu) : b1 > 0 si le temps economise se traduit en revenus
- Q3 (attitude) : b1 > 0 si la formation augmente l'attitude entrepreneuriale
- Si b1 ≈ b2 : l'effet vient de la formation, pas du foyer
- Si b1 > b2 : le foyer ajoute un effet au-dela de la formation